In [2]:
import torch
import gc

# Очищаем кэш GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# Принудительный сборщик мусора
gc.collect()

# Проверяем результат
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory cached: 0.00 GB


In [3]:
from transformers import BitsAndBytesConfig 
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16, 
#     bnb_4bit_use_double_quant=False, 
#     bnb_4bit_quant_type="nf4",  
#     # llm_int8_enable_fp32_cpu_offload=True,
#     bnb_4bit_quant_storage=torch.uint8
# )

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,  # Ключевой параметр
    llm_int8_threshold=6.0
)


In [8]:
MODEL = "Qwen/Qwen1.5-MoE-A2.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, 
                                            #  device_map='cuda',
                                            # padding_side="left",
                                             )
# model = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True, 
#                                             device_map='auto',
#                                             torch_dtype=torch.float16,
#                                             low_cpu_mem_usage=True,
#                                             use_cache=True,
#                                             quantization_config=quantization_config
                                             
#                        )
model = AutoModelForCausalLM.from_pretrained(
    MODEL, 
    trust_remote_code=True, 
    device_map="auto",
    quantization_config=quantization_config,
    # torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    # offload_buffers=True
)

Loading checkpoint shards: 100%|██████████| 8/8 [00:21<00:00,  2.73s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [9]:
import torch
import torch.nn.utils.prune as prune
import torch_pruning as tp

In [10]:
def count_parameters(model):
    """Подсчет общего количества параметров"""
    return sum(p.numel() for p in model.parameters())

def count_non_zero_params(model):
    """Подсчет ненулевых параметров (для оценки разреженности)"""
    non_zero = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        non_zero += torch.count_nonzero(param).item()
    return non_zero, total

def get_model_memory_usage(model):
    """Оценка использования памяти моделью в MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb


In [11]:
from torch.utils.data import DataLoader
from datasets import load_dataset

In [12]:
def load_test_dataset(dataset_name='imdb', num_samples=100, max_length=128):
    """
    Загрузка небольшого датасета для тестирования
    """
    dataset = load_dataset(dataset_name, split='test')
    
    # Берем только небольшое количество образцов для тестирования
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"]
    )
    
    # Создаем DataLoader
    from torch.utils.data import DataLoader
    
    class SimpleDataset(torch.utils.data.Dataset):
        def __init__(self, hf_dataset):
            self.hf_dataset = hf_dataset
        
        def __len__(self):
            return len(self.hf_dataset)
        
        def __getitem__(self, idx):
            item = self.hf_dataset[idx]
            return {
                "input_ids": torch.tensor(item['input_ids']),
                "attention_mask": torch.tensor(item['attention_mask']),
                "labels": torch.tensor(item['input_ids'])  # Для causal LM
            }
    
    dataset_obj = SimpleDataset(tokenized_dataset)
    dataloader = DataLoader(dataset_obj, batch_size=2, shuffle=False)
    
    return dataloader

In [10]:
def apply_adaptpruner_pruning(model, pruning_ratio=0.3, example_inputs=None):
    try:
        import torch_pruning as tp
    except ImportError:
        return model
    
    
    # Определяем слои для исключения из обрезки
    ignored_layers = []
    
    for name, module in model.named_modules():
        # Исключаем критические слои
        if any(x in name for x in ['embed_tokens', 'lm_head', 'embedding']):
            ignored_layers.append(module)
            continue
        
        # Для MoE исключаем экспертные слои и гейты (они критически важны)
        if any(x in name.lower() for x in ['expert', 'gate', 'router', 'moe']):
            ignored_layers.append(module)
            continue
    
    # Исключаем слои нормализации
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.LayerNorm, torch.nn.BatchNorm1d)):
            if module not in ignored_layers:
                ignored_layers.append(module)
    
    # for name, module in model.named_modules():
    #     if 'Linear8bitLt' in str(type(module)):
    #         if module not in ignored_layers:
    #             ignored_layers.append(module)
    
    
    # Используем переданные example_inputs или создаем новые
    if example_inputs is None:
        device = next(model.parameters()).device
        batch_size = 2
        seq_length = 64
        example_inputs = {
            "input_ids": torch.randint(0, tokenizer.vocab_size, (batch_size, seq_length)).to(device),
            "attention_mask": torch.ones(batch_size, seq_length).to(device),
        }
    else:
        # Убираем 'labels' если он есть
        if 'labels' in example_inputs:
            example_inputs = {k: v for k, v in example_inputs.items() if k != 'labels'}
    
    # Создаем враппер для модели
    class ModelWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        
        def forward(self, input_ids, attention_mask):
            return self.model(input_ids=input_ids, attention_mask=attention_mask)
    
    wrapped_model = ModelWrapper(model)
    
    try:
        # Настраиваем важность (magnitude-based)
        importance = tp.importance.MagnitudeImportance(p=2)
        
        # Структурированная обрезка
        pruner = tp.pruner.MetaPruner(
            wrapped_model,
            example_inputs,
            importance=importance,
            pruning_ratio=pruning_ratio,
            ignored_layers=ignored_layers,
            round_to=8,
            iterative_steps=1,
            # Важно: не обрезаем последнее измерение
            ch_sparsity=False,
        )
        
        # Применяем обрезку
        pruner.step()
        
        # Получаем обрезанную модель
        pruned_model = wrapped_model.model
        
        # Очищаем память
        torch.cuda.empty_cache()
        gc.collect()
        
        
        return pruned_model
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return model

In [16]:
import torch
import torch_pruning as tp
import gc

def apply_adaptpruner_pruning_safe(model, pruning_ratio=0.3, example_inputs=None, use_structured=True):

    # Определяем слои для игнорирования по имени или типу
    ignored_layers = []
    for name, module in model.named_modules():
        # Критические слои: embedding, lm_head, gating
        if any(x in name.lower() for x in ['embed', 'lm_head', 'gate', 'expert', 'router', 'moe']):
            ignored_layers.append(module)
        # Нормализация
        if isinstance(module, (torch.nn.LayerNorm, torch.nn.BatchNorm1d)):
            ignored_layers.append(module)

    # Создаём пример входа
    if example_inputs is None:
        device = next(model.parameters()).device
        batch_size = 2
        seq_length = 64
        example_inputs = {
            "input_ids": torch.randint(0, 1000, (batch_size, seq_length), device=device),
            "attention_mask": torch.ones(batch_size, seq_length, device=device)
        }
    else:
        # убираем labels
        example_inputs = {k:v for k,v in example_inputs.items() if k != "labels"}

    # Враппер для модели
    class ModelWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, input_ids, attention_mask):
            return self.model(input_ids=input_ids, attention_mask=attention_mask)

    wrapped_model = ModelWrapper(model)

    class SafeMagnitudeImportance(tp.importance.MagnitudeImportance):
        def step(self, module, weight_name='weight'):
            weight = getattr(module, weight_name, None)
            if weight is None or weight.device.type == 'meta':
                # Возвращаем нулевой тензор той же формы на CPU
                fake_weight = getattr(module, weight_name, None)
                shape = fake_weight.shape if fake_weight is not None else (1,)
                return torch.zeros(shape, device='cpu')
            return super().step(module, weight_name)

    try:
        importance = SafeMagnitudeImportance(p=2)

        if use_structured:
            # Структурированная обрезка
            pruner = tp.pruner.MetaPruner(
                wrapped_model,
                example_inputs,
                importance=importance,
                pruning_ratio=pruning_ratio,
                ignored_layers=ignored_layers,
                round_to=8,
                iterative_steps=1,
                ch_sparsity=False,
            )
        else:
            # Неструктурированная
            pruner = tp.pruner.UnstructuredPruner(
                wrapped_model,
                example_inputs,
                importance=importance,
                pruning_ratio=pruning_ratio,
                ignored_layers=ignored_layers,
            )

        pruner.step()

        # Очищаем память
        torch.cuda.empty_cache()
        gc.collect()

        return wrapped_model.model

    except Exception as e:
        import traceback
        traceback.print_exc()
        print("Ошибка при pruning, возвращаем исходную модель")
        return model

In [17]:
def convert_attention_layers(model):
    import bitsandbytes as bnb
    
    for name, module in model.named_modules():
        # Ищем attention проекции
        if any(x in name for x in ['q_proj', 'k_proj', 'v_proj', 'o_proj']):
            if isinstance(module, bnb.nn.Linear8bitLt):
                try:
                    # Получаем информацию о слое
                    in_features = module.in_features
                    out_features = module.out_features
                    has_bias = module.bias is not None
                    
                    # Создаем новый Linear
                    new_layer = torch.nn.Linear(
                        in_features,
                        out_features,
                        bias=has_bias,
                        dtype=torch.float16
                    )
                    
                    # Получаем веса из 8-битного слоя
                    # Веса хранятся в module.weight (uint8) и module.weight_quant_state
                    if hasattr(module, 'weight') and module.weight is not None:
                        # Деквантизируем веса
                        if hasattr(module, 'weight_quant_state'):
                            # Используем bitsandbytes для деквантизации
                            weight_data = bnb.functional.dequantize_8bit(
                                module.weight, 
                                module.weight_quant_state
                            )
                        else:
                            # Просто конвертируем в float
                            weight_data = module.weight.float()
                        
                        # Копируем веса
                        new_layer.weight.data = weight_data.to(torch.float16)
                    
                    # Копируем bias если есть
                    if has_bias and module.bias is not None:
                        new_layer.bias.data = module.bias.float().to(torch.float16)
                    
                    # Заменяем слой
                    parent_name = '.'.join(name.split('.')[:-1])
                    child_name = name.split('.')[-1]
                    
                    # Получаем родительский модуль
                    parent = model
                    for part in parent_name.split('.'):
                        if part:
                            parent = getattr(parent, part)
                    
                    # Заменяем
                    setattr(parent, child_name, new_layer)
                    
                    
                except Exception as e:
                    continue
    
    return model

In [51]:
x = model.model.layers[0].mlp.experts[0].gate_proj

In [52]:
??x

Signature:      x(*args, **kwargs)
Type:           Linear8bitLt
String form:    Linear8bitLt(in_features=2048, out_features=1408, bias=False)
File:           ~/projects/FedCore/.venv/lib/python3.10/site-packages/bitsandbytes/nn/modules.py
Source:        
class Linear8bitLt(nn.Linear):
    """
    This class is the base module for the [LLM.int8()](https://arxiv.org/abs/2208.07339) algorithm.
    To read more about it, have a look at the paper.

    In order to quantize a linear layer one should first load the original fp16 / bf16 weights into
    the Linear8bitLt module, then call `int8_module.to("cuda")` to quantize the fp16 weights.

    Example:

    ```python
    import torch
    import torch.nn as nn

    import bitsandbytes as bnb
    from bnb.nn import Linear8bitLt

    fp16_model = nn.Sequential(
        nn.Linear(64, 64),
        nn.Linear(64, 64)
    )

    int8_model = nn.Sequential(
        Linear8bitLt(64, 64, has_fp16_weights=False),
        Linear8bitLt(64, 64, has_fp16_w

In [18]:
# fedcore_train_data, fedcore_test_data = load_test_dataset('imdb', batch_size=4, num_samples=None)

params_before = count_parameters(model)
print(f"Total parameters BEFORE pruning: {params_before:,}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)
# model = convert_attention_layers(model)
print(model)
batch_size = 2
seq_length = 64
example_inputs = {
        "input_ids": torch.randint(0, tokenizer.vocab_size, (batch_size, seq_length)).to(device),
        "attention_mask": torch.ones(batch_size, seq_length).to(device),
        "labels": torch.randint(0, tokenizer.vocab_size, (batch_size, seq_length)).to(device)
    }
    
pruned_model = apply_adaptpruner_pruning_safe(
        model, 
        pruning_ratio=0.5,
        example_inputs=example_inputs
    )
print(pruned_model)
params_after = count_parameters(pruned_model)

print(f"Parameters before: {params_before:,}")
print(f"Parameters after: {params_after:,}")
print(f"Parameters removed: {params_before - params_after:,}")
print(f"Reduction: {(params_before - params_after)/params_before*100:.2f}%")

Total parameters BEFORE pruning: 14,315,784,192
Qwen2MoeForCausalLM(
  (model): Qwen2MoeModel(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-23): 24 x Qwen2MoeDecoderLayer(
        (self_attn): Qwen2MoeSdpaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2MoeRotaryEmbedding()
        )
        (mlp): Qwen2MoeSparseMoeBlock(
          (gate): Linear8bitLt(in_features=2048, out_features=60, bias=False)
          (experts): ModuleList(
            (0-59): 60 x Qwen2MoeMLP(
              (gate_proj): Linear8bitLt(in_features=2048, out_features=1408, bias=False)
              (up_proj): Linear8bitLt(in_features=2048, out_features=1408, bias=False)
 

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.model.layers.14.input_layernorm.weight', 'model.model.layers.19.input_layernorm.weight', 'model.model.layers.22.post_attention_layernorm.weight', 'model.model.layers.12.post_attention_layernorm.weight', 'model.model.layers.14.post_attention_layernorm.weight', 'model.model.layers.8.post_attention_layernorm.weight', 'model.model.layers.0.post_attention_layernorm.weight', 'model.model.layers.1.post_attention_layernorm.weight', 'model.model.layers.5.post_attention_layernorm.weight', 'model.model.layers.6.input_layernorm.weight', 'model.model.layers.6.post_attention_layernorm.weight', 'model.model.layers.7.input_layernorm.weight', 'model.model.layers.7.post_attention_layernorm.weight', 'model.model.layers.10.post_attention_layernorm.weight', 'model.model.layers.11.post_attention_layernorm.weight', 'model.model.layers.2.input_layernorm.weig

Ошибка при pruning, возвращаем исходную модель
Qwen2MoeForCausalLM(
  (model): Qwen2MoeModel(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-23): 24 x Qwen2MoeDecoderLayer(
        (self_attn): Qwen2MoeSdpaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2MoeRotaryEmbedding()
        )
        (mlp): Qwen2MoeSparseMoeBlock(
          (gate): Linear8bitLt(in_features=2048, out_features=60, bias=False)
          (experts): ModuleList(
            (0-59): 60 x Qwen2MoeMLP(
              (gate_proj): Linear8bitLt(in_features=2048, out_features=1408, bias=False)
              (up_proj): Linear8bitLt(in_features=2048, out_features=1408, bias=False)
  

Traceback (most recent call last):
  File "/tmp/ipykernel_974/4046418046.py", line 75, in apply_adaptpruner_pruning_safe
    pruner.step()
  File "/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/pruner/algorithms/base_pruner.py", line 282, in step
    for group in self._prune():
  File "/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 40, in generator_context
    response = gen.send(None)
  File "/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/pruner/algorithms/base_pruner.py", line 490, in _prune
    dim_imp = imp.cpu()
NotImplementedError: Cannot copy out of meta tensor; no data!
